# `build_design.py` geometry visualization

Two grounded circular transmons (`CircTransmon`) joined by a single **fixed
capacitive coupler** — an interdigitated finger capacitor with fully rounded
metal (`CapNInterdigitalRound`, no sharp corners anywhere). Each comb of the
capacitor is a direct galvanic extension of one transmon island: its lead
flares out and blends tangentially into the round pad (no CPW / claws / routing).
The two combs never touch, so `Q1` and `Q2` stay on separate nets and the
inter-comb capacitance is the coupling capacitance.

Same design as `build_design.py`, split into cells with a `MetalGUI` so the
geometry can be inspected interactively.

In [1]:
from qiskit_metal import designs, Dict, MetalGUI
from qiskit_metal.qlibrary.qubits.circle_transmon import CircTransmon
from qiskit_metal.qlibrary.couplers.cap_n_interdigital_round import CapNInterdigitalRound

from build_design import (
    L_Q1_NH, L_Q2_NH, CJ_Q1_FF, CJ_Q2_FF,
    JJ_WIDTH_UM, JJ_SIM_GAP_UM, JJ_ANGLE_DEG,
    PAD_RADIUS_UM, PAD_GAP_UM,
    Q1_NAME, Q2_NAME, COUPLER_NAME,
    CHIP_SIZE_Z_UM, CHIP_MARGIN_FACTOR,
    _design_bounds_mm, _transmon_offset_um,
    _transmon_options, _coupler_options,
)

In [2]:
design = designs.DesignPlanar({}, overwrite_enabled=True)
gui = MetalGUI(design)

design.variables['cpw_width'] = '20um'
design.variables['cpw_gap'] = '12.25um'

## Transmons

Two identical `CircTransmon`s. `D` (the `|pos_x|` of each) is derived from the
coupler geometry so each pad edge overlaps the coupler's lead stub — see
`_transmon_offset_um()` in `build_design.py`. The single JJ points south
(`jj_angle = 270`), away from the east/west coupler.

In [3]:
D = _transmon_offset_um()
print(f"transmon offset |pos_x| = {D:.1f} um")

q1 = CircTransmon(design, Q1_NAME, options=_transmon_options(-D, L_Q1_NH, CJ_Q1_FF))
q2 = CircTransmon(design, Q2_NAME, options=_transmon_options(+D, L_Q2_NH, CJ_Q2_FF))

gui.rebuild()
gui.autoscale()

transmon offset |pos_x| = 227.0 um


## Interdigital capacitive coupler

`CapNInterdigitalRound`, rotated 90 deg so its comb-separation axis lies along
global x: comb `a` abuts `Q2` (+x), comb `b` abuts `Q1` (-x). Every finger tip is
a semicircle and every finger-root corner is filleted.

Because it is told the transmon pad geometry (`island_radius` / `island_gap` /
`island_center_offset`), it runs in **circular-island mode**: each comb's lead is
a tapered foot that flares out and blends tangentially into the round pad (no
sharp point where lead meets pad), and the rectangular ground moat is merged with
each transmon pocket with the re-entrant cusps filleted (`etch_blend_radius`) —
so the combined cutout has no inward spikes.

All coupler numbers live in `build_design.py` (`CPL_*` constants,
`_coupler_options()`); edit them there. Capacitance grows with
`finger_count` / `finger_length` and shrinks as `finger_gap` /
`finger_tip_gap` grow.

In [4]:
try:
    coupler.delete()
except NameError:
    pass

coupler = CapNInterdigitalRound(design, COUPLER_NAME, options=_coupler_options())

gui.rebuild()
gui.autoscale()

## Size the chip from the drawn geometry

`size_x` / `size_y` / `center_*` are taken from the geometry's own bounding box
(`_design_bounds_mm`) times `CHIP_MARGIN_FACTOR`; `size_z` is fixed.

In [ ]:
minx, miny, maxx, maxy = _design_bounds_mm(design)
bcx, bcy = (minx + maxx) / 2, (miny + maxy) / 2
bsx, bsy = (maxx - minx), (maxy - miny)

design.chips.main.size.size_x = f'{bsx * CHIP_MARGIN_FACTOR}mm'
design.chips.main.size.size_y = f'{bsy * CHIP_MARGIN_FACTOR}mm'
design.chips.main.size.size_z = f'{CHIP_SIZE_Z_UM}um'
design.chips.main.size.center_x = f'{bcx}mm'
design.chips.main.size.center_y = f'{bcy}mm'

gui.rebuild()
gui.autoscale()

print("components:", list(design.components.keys()))
print(f"design bounds (mm): X=[{minx:.4f}, {maxx:.4f}]  Y=[{miny:.4f}, {maxy:.4f}]")
print(f"chip (mm): {design.chips.main.size.size_x} x {design.chips.main.size.size_y}")
print("junction rows:", len(design.qgeometry.tables['junction']))

components: ['Q1', 'Q2', 'coupler']
design bounds (mm): X=[-0.2650, 0.2650]  Y=[-0.1000, 0.1000]
chip (mm): 1.59mm x 0.6000000000000001mm
junction rows: 2


## Cross-check against `build_design.build_design()`

The function form must produce the same geometry as the cells above.

In [6]:
import numpy as np
from build_design import build_design

d2 = build_design()
d2.rebuild()
assert list(d2.components.keys()) == list(design.components.keys())
assert len(d2.qgeometry.tables['junction']) == len(design.qgeometry.tables['junction'])
assert np.allclose(d2.qgeometry.tables['poly'].total_bounds,
                   design.qgeometry.tables['poly'].total_bounds, atol=1e-9)
print("build_design() matches the cells above:", list(d2.components.keys()))

build_design() matches the cells above: ['Q1', 'Q2', 'coupler']
